# 4 — Neuron List

Compares universal circuits against token checker circuits to classify neurons
as concept-only, shared, or token-only.

Loops over all 9 epsilon x consistency settings in a single run.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"

EPSILONS = [0.001, 0.1, 0.5]
CONSISTENCIES = [0.2, 0.5, 0.8]
LAYERS = list(range(28))
OBJECT_TYPE = "both"    # "ast", "builtin", or "both"
N_LAYERS = 28

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Grid: {len(EPSILONS)} eps x {len(CONSISTENCIES)} cons = {len(EPSILONS)*len(CONSISTENCIES)} combos")
print(f"Layers: {len(LAYERS)}, type: {OBJECT_TYPE}")

In [ ]:
# Cell 2 – Helper functions

def match_type(ds_name):
    if OBJECT_TYPE == "both":
        return True
    return ds_name.startswith(OBJECT_TYPE)


def load_universal_masks(obj_file):
    """Load universal masks from HDF5."""
    universal = {}
    path = os.path.join(DATA_DIR, obj_file)
    if not os.path.exists(path):
        return None
    with h5py.File(path, "r") as f:
        um = f["universal_masks"]
        for lid in LAYERS:
            lk = f"layer_{lid}"
            if lk not in um:
                continue
            for ds_name in um[lk]:
                if not match_type(ds_name):
                    continue
                if ds_name not in universal:
                    universal[ds_name] = {}
                universal[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
    return universal


def load_checker_masks(chk_file):
    """Load checker masks from HDF5."""
    checker = {}
    path = os.path.join(DATA_DIR, chk_file)
    if not os.path.exists(path):
        return None
    with h5py.File(path, "r") as f:
        tcm = f["token_checker_masks"]
        for lid in LAYERS:
            lk = f"layer_{lid}"
            if lk not in tcm:
                continue
            for ds_name in tcm[lk]:
                if not match_type(ds_name):
                    continue
                if ds_name not in checker:
                    checker[ds_name] = {}
                checker[ds_name][lid] = np.array(tcm[lk][ds_name], dtype=bool)
    return checker


def detect_neuron_dim(masks):
    """Auto-detect neuron dimension from first mask."""
    for layers in masks.values():
        for mask in layers.values():
            return len(mask)
    return 3584


print("Helpers defined.")

In [ ]:
# Cell 3 – Loop over all 9 settings

saved_files = []

for eps in EPSILONS:
    for cons in CONSISTENCIES:
        obj_file = f"{PREFIX}3_object_masks_eps{eps}_cons{cons}.h5"
        chk_file = f"{PREFIX}3_checker_masks_eps{eps}_cons{cons}.h5"

        print(f"\n--- eps={eps}, cons={cons} ---")

        universal = load_universal_masks(obj_file)
        if universal is None:
            print(f"  SKIP: {obj_file} not found")
            continue

        checker = load_checker_masks(chk_file)
        if checker is None:
            print(f"  SKIP: {chk_file} not found")
            continue

        testable = sorted(set(universal.keys()) & set(checker.keys()))
        print(f"  Universal: {len(universal)}, Checker: {len(checker)}, Testable: {len(testable)}")

        if len(testable) == 0:
            print("  SKIP: no testable objects")
            continue

        neuron_dim = detect_neuron_dim(universal)

        rows = []
        for obj in testable:
            for lid in LAYERS:
                A = universal[obj].get(lid, np.zeros(neuron_dim, dtype=bool))
                B = checker[obj].get(lid, np.zeros(neuron_dim, dtype=bool))

                concept_only = np.where(A & ~B)[0].tolist()
                both = np.where(A & B)[0].tolist()
                token_only = np.where(~A & B)[0].tolist()

                rows.append({
                    "object": obj,
                    "layer": lid,
                    "n_concept_only": len(concept_only),
                    "n_both": len(both),
                    "n_token_only": len(token_only),
                    "concept_only": str(concept_only),
                    "both": str(both),
                    "token_only": str(token_only),
                })

        df = pd.DataFrame(rows)
        out_name = f"{PREFIX}4_neuron_list_eps{eps}_cons{cons}_all_layers_{OBJECT_TYPE}.csv"
        out_path = os.path.join(DATA_DIR, out_name)
        df.to_csv(out_path, index=False)
        saved_files.append(out_name)
        print(f"  Saved: {out_name} ({len(df)} rows, {len(testable)} objects x {len(LAYERS)} layers)")

print(f"\nDone: {len(saved_files)} / {len(EPSILONS)*len(CONSISTENCIES)} files saved")

In [ ]:
# Cell 4 – Summary

print("SAVED FILES:")
for f in saved_files:
    print(f"  {f}")

print(f"\n4 complete.")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")